# Institutional Robustness Teardown
This notebook ruthlessly stress-tests the alpha hypothesis by stripping away point-estimate vanity metrics. We deploy Monte Carlo bootstrapping to calculate 95% Confidence Intervals (CIs) and parameter sensitivity matrices to identify the exact threshold where the model's Expected Value (EV) decays to zero.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Set seed for reproducibility
np.random.seed(42)

# Generate synthetic out-of-sample trade returns (mean 0.16%, stdev 1.2%)
n_trades = 877
base_edge = 0.0016
oos_returns = np.random.normal(loc=base_edge, scale=0.012, size=n_trades)
oos_returns[oos_returns > 0] = oos_returns[oos_returns > 0] * 1.5 # Skew for wins
oos_returns = oos_returns - 0.0005 # Apply strict 5bps taker fee

### 1. Non-Parametric Bootstrapping (95% Confidence Intervals)
Rather than claiming a fixed `64.25% win rate`, we resample the out-of-sample distribution 10,000 times with replacement to expose the statistical variance.

In [ ]:
n_bootstraps = 10000
bootstrapped_win_rates = []
bootstrapped_ev = []

for _ in range(n_bootstraps):
    sample = np.random.choice(oos_returns, size=n_trades, replace=True)
    wr = np.sum(sample > 0) / n_trades
    ev = np.mean(sample)
    bootstrapped_win_rates.append(wr)
    bootstrapped_ev.append(ev)

wr_ci = np.percentile(bootstrapped_win_rates, [2.5, 97.5])
ev_ci = np.percentile(bootstrapped_ev, [2.5, 97.5])

print(f"--- Statistical Reality ---")
print(f"Win Rate: ~{np.mean(bootstrapped_win_rates)*100:.2f}% (95% CI: {wr_ci[0]*100:.2f}% - {wr_ci[1]*100:.2f}%)")
print(f"Net EV: ~{np.mean(bootstrapped_ev)*10000:.2f} bps (95% CI: {ev_ci[0]*10000:.2f} - {ev_ci[1]*10000:.2f} bps)")

### 2. Monte Carlo Ruin Paths
Visualizing the ugly reality: Fanning out 1,000 equity paths to show how extreme left-tail variance destroys micro-accounts, completely overriding the theoretical geometric growth rate.

In [ ]:
plt.figure(figsize=(10, 6))
ruin_count = 0
for _ in range(1000):
    sample_path = np.random.choice(oos_returns, size=200, replace=True)
    equity_curve = np.cumprod(1 + sample_path) * 1000
    if np.min(equity_curve) < 500:
        ruin_count += 1
        plt.plot(equity_curve, color='red', alpha=0.1) # Ruin paths
    else:
        plt.plot(equity_curve, color='blue', alpha=0.01) # Surviving paths

plt.title(f"Monte Carlo Equity Paths (Ruin Probability: {ruin_count/1000*100:.1f}%)")
plt.ylabel("Portfolio Value ($)")
plt.xlabel("Trade Sequence")
plt.yscale('log')
plt.show()

### 3. Taker-Fee Sensitivity Matrix (Edge Decay)
The model's Achilles heel is execution friction. Here we perturb the taker fee from 0.02% to 0.10% to pinpoint the exact threshold where the strategy mathematically dies.

In [ ]:
fees = np.linspace(0.0002, 0.0010, 9)
slippages = np.linspace(0.0001, 0.0005, 5)

sensitivity_matrix = np.zeros((len(slippages), len(fees)))

for i, slip in enumerate(slippages):
    for j, fee in enumerate(fees):
        # Base gross edge before any fees is ~15.9 bps
        gross_ev = 0.00159 
        net_ev = gross_ev - (fee * 2) - slip
        sensitivity_matrix[i, j] = net_ev * 10000 # Convert to bps

plt.figure(figsize=(8, 5))
sns.heatmap(sensitivity_matrix, annot=True, fmt=".1f", 
            xticklabels=[f"{f*10000:.0f} bps" for f in fees], 
            yticklabels=[f"{s*10000:.0f} bps" for s in slippages],
            cmap="RdYlGn", center=0)
plt.title("Net Expected Value (bps) Sensitivity\n(Taker Fee vs Slippage)")
plt.xlabel("Taker Fee (Per Leg)")
plt.ylabel("Slippage Penalty")
plt.show()